# GNN Starter for Kaggle Playground March Churn Competition
For Churn, GNN (graph neural networks) are strong because the graph contains information about which customers are related to which customers. The prediction of Churn can be inferred from what customers' friends are doing. For example, if everyone's friends are leaving a particular provider, then that customer will most likely churn too. This dataset doesn't provide explicit feature to determine customer friends but we can deduce them using KNN for to discover similar customers.

This notebook uses Nvidia cuML KNN as proxy to determine customer's friends. Then it trains a GNN to predict classifcation of nodes where nodes are customers. The GNN uses neighborhood nodes (i.e. friends) to help predict each customer's probability of Churn! 

This model is diverse from other models and we demonstate this via GPU Hill Climbing. Hill climbing selects this model even though an ensemble already has XGBost, Logistic Regression, and PyTorch MLP.

The following code was written by ChatGPT with prompting from Chris. There is a discusion about this notebook [here][1]

[1]: https://www.kaggle.com/competitions/playground-series-s6e3/discussion/680622

In [1]:
!pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.4 MB/s eta 0:00:00


In [2]:
# ============================================================
# TELCO CHURN — GNN SIMPLE 25 FEATURES (GraphSAGE, FAST BATCHED)
#
# Node features (same simple 25 features):
#   - 16 original categorical columns
#   - 3 snapped numeric->categorical proxy columns
#   - 3 numeric rare-flag categorical columns
#   - 3 numeric columns as direct inputs
#
# Graph construction:
#   - OHE of 16 original categorical columns
#   - PLUS 3 numeric columns
#   - numeric columns are StandardScaler normalized
#   - then multiplied by GRAPH_NUMERIC_MULTIPLIER
#
# Saves:
#   - oof_gnn_v{VER}.npy
#   - pred_gnn_v{VER}.npy
# ============================================================

import numpy as np
import pandas as pd
import gc
import time
import random
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from cuml.neighbors import NearestNeighbors

from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv

# -----------------------------
# CONFIG
# -----------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TARGET = "Churn"
DROP_COLS = ["customerID", "id"]

VER = 1909

SEED = 42
N_FOLDS = 5
RARE_MIN_COUNT = 25

# Faster graph/training settings
K = 8
EPOCHS = 5
PATIENCE = 8
VAL_EVERY = 1
MIN_EPOCHS = 6

GRAPH_NUMERIC_MULTIPLIER = 3.0

# Manual mini-batch settings
BATCH_SIZE = 8192
INFER_BATCH_SIZE = 16384
FANOUTS = [6, 4]   # 2 hops for 2 GraphSAGE layers

USE_AMP = True

BASE_NUMS = ["tenure", "MonthlyCharges", "TotalCharges"]
BASE_CATS = [
    "gender", "SeniorCitizen", "Partner", "Dependents", "PhoneService",
    "MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup",
    "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies",
    "Contract", "PaperlessBilling", "PaymentMethod",
]

CAT_PROXY = [f"{c}__cat" for c in BASE_NUMS]
CAT_RARE = [f"{c}__is_rare" for c in BASE_NUMS]

CAT_COLS = BASE_CATS + CAT_PROXY + CAT_RARE   # 22 categorical node features
NUM_COLS = BASE_NUMS[:]                       # 3 numeric node features

GRAPH_CAT_COLS = BASE_CATS[:]                 # 16 categoricals for graph
GRAPH_NUM_COLS = BASE_NUMS[:]                 # 3 numerics for graph

# -----------------------------
# REPRO
# -----------------------------
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# -----------------------------
# LOAD
# -----------------------------
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/test.csv")
print("train:", train.shape, "test:", test.shape)

for c in DROP_COLS:
    if c in train.columns:
        train.drop(columns=[c], inplace=True)
    if c in test.columns:
        test.drop(columns=[c], inplace=True)

if TARGET not in train.columns:
    raise KeyError(f"TARGET column '{TARGET}' not found in train.csv")

y = train[TARGET].astype(str).str.strip().map({"Yes": 1, "No": 0}).values.astype(np.float32)

# -----------------------------
# CLEAN BASE DATA
# -----------------------------
def clean_totalcharges(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "TotalCharges" in df.columns:
        df["TotalCharges"] = df["TotalCharges"].astype(str).str.strip()
        df["TotalCharges"] = df["TotalCharges"].replace("", np.nan)
        df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
    return df

train = clean_totalcharges(train)
test = clean_totalcharges(test)

def preprocess_base(train_df: pd.DataFrame, test_df: pd.DataFrame):
    tr = train_df.copy()
    te = test_df.copy()

    # numeric cleanup
    for df in (tr, te):
        for c in BASE_NUMS:
            df[c] = pd.to_numeric(df[c], errors="coerce").astype(np.float32)

    # categorical cleanup
    for df in (tr, te):
        for c in BASE_CATS:
            df[c] = df[c].astype(str).fillna("missing").str.strip()

    # fill numeric NaNs using TRAIN medians only
    for c in BASE_NUMS:
        med = float(np.nanmedian(tr[c].to_numpy(np.float32)))
        if not np.isfinite(med):
            med = 0.0
        tr[c] = tr[c].fillna(med).astype(np.float32)
        te[c] = te[c].fillna(med).astype(np.float32)

    return tr, te

train_base, test_base = preprocess_base(train, test)

# -----------------------------
# SIMPLE 25 FEATURE ENGINEERING
# -----------------------------
def build_numeric_snapper(train_series: pd.Series, rare_min_count: int):
    """
    From TRAIN data only:
      - frequent values = count >= rare_min_count
      - rare values snapped to nearest frequent value
      - rare flag returned too
    """
    s = pd.to_numeric(train_series, errors="coerce").astype(np.float32)
    vc = pd.Series(s).value_counts(dropna=False)

    frequent_vals = vc[vc >= rare_min_count].index.values
    frequent_vals = np.array([v for v in frequent_vals if pd.notna(v)], dtype=np.float32)

    if frequent_vals.size == 0:
        frequent_vals = np.array(pd.Series(s.dropna()).unique(), dtype=np.float32)

    frequent_vals = np.sort(frequent_vals)
    frequent_set = set(frequent_vals.tolist())

    def transform(series_any: pd.Series):
        x = pd.to_numeric(series_any, errors="coerce").astype(np.float32).values
        is_nan = np.isnan(x)

        is_rare = np.ones_like(x, dtype=np.int32)
        for i, v in enumerate(x):
            if np.isnan(v):
                is_rare[i] = 1
            else:
                is_rare[i] = 0 if float(v) in frequent_set else 1

        x_snapped = x.copy()
        if frequent_vals.size > 0:
            idx_snap = np.where((~is_nan) & (is_rare == 1))[0]
            if idx_snap.size > 0:
                v = x[idx_snap]
                pos = np.searchsorted(frequent_vals, v)
                pos = np.clip(pos, 0, len(frequent_vals) - 1)

                left = np.clip(pos - 1, 0, len(frequent_vals) - 1)
                right = pos

                left_vals = frequent_vals[left]
                right_vals = frequent_vals[right]

                choose_right = (np.abs(v - right_vals) <= np.abs(v - left_vals))
                nearest = np.where(choose_right, right_vals, left_vals)
                x_snapped[idx_snap] = nearest.astype(np.float32)

        return x_snapped.astype(np.float32), is_rare.astype(np.int32)

    return transform

def make_simple25_features(train_df: pd.DataFrame, test_df: pd.DataFrame):
    tr = train_df.copy()
    te = test_df.copy()

    for col in BASE_NUMS:
        snapper = build_numeric_snapper(tr[col], rare_min_count=RARE_MIN_COUNT)

        tr_snap, tr_israre = snapper(tr[col])
        te_snap, te_israre = snapper(te[col])

        tr[f"{col}__cat"] = pd.Series(tr_snap).astype(str).values
        te[f"{col}__cat"] = pd.Series(te_snap).astype(str).values

        tr[f"{col}__is_rare"] = pd.Series(tr_israre).astype(str).values
        te[f"{col}__is_rare"] = pd.Series(te_israre).astype(str).values

    for df in (tr, te):
        for c in CAT_COLS:
            df[c] = df[c].astype(str).fillna("missing")

    return tr, te

train_fe, test_fe = make_simple25_features(train_base, test_base)

# -----------------------------
# ENCODE NODE CATEGORICALS
# -----------------------------
def encode_cats(train_df, test_df, cat_cols):
    tr_codes = []
    te_codes = []
    cardinals = []

    for c in cat_cols:
        all_vals = pd.concat(
            [train_df[c].astype(str), test_df[c].astype(str)],
            axis=0,
            ignore_index=True
        )
        uniq = all_vals.unique().tolist()
        mapping = {v: i for i, v in enumerate(uniq)}

        tr_c = train_df[c].astype(str).map(mapping).fillna(0).astype(np.int64).values
        te_c = test_df[c].astype(str).map(mapping).fillna(0).astype(np.int64).values

        tr_codes.append(tr_c)
        te_codes.append(te_c)
        cardinals.append(len(mapping))

    Xc_tr = np.stack(tr_codes, axis=1) if len(tr_codes) else np.zeros((len(train_df), 0), dtype=np.int64)
    Xc_te = np.stack(te_codes, axis=1) if len(te_codes) else np.zeros((len(test_df), 0), dtype=np.int64)

    return Xc_tr, Xc_te, cardinals

Xc_train, Xc_test, cat_cardinals = encode_cats(train_fe, test_fe, CAT_COLS)

# -----------------------------
# NODE NUMERIC MATRIX (same simple 25 features)
# -----------------------------
for c in NUM_COLS:
    train_fe[c] = pd.to_numeric(train_fe[c], errors="coerce").fillna(0).astype(np.float32)
    test_fe[c]  = pd.to_numeric(test_fe[c], errors="coerce").fillna(0).astype(np.float32)

Xn_train = train_fe[NUM_COLS].values.astype(np.float32)
Xn_test  = test_fe[NUM_COLS].values.astype(np.float32)

node_scaler = StandardScaler()
Xn_train = node_scaler.fit_transform(Xn_train).astype(np.float32)
Xn_test  = node_scaler.transform(Xn_test).astype(np.float32)

Xn_all = np.vstack([Xn_train, Xn_test])
Xc_all = np.vstack([Xc_train, Xc_test])

n_train = len(train_fe)
n_test = len(test_fe)
n_all = n_train + n_test

print("Using same simple 25 node features:")
print("NUM_COLS:", NUM_COLS)
print("CAT_COLS:", CAT_COLS)
print("Xn_all:", Xn_all.shape, "Xc_all:", Xc_all.shape)

# ============================================================
# BUILD KNN GRAPH ON:
#   OHE(16 base cats) + standardized(3 numerics) * multiplier
# ============================================================
graph_train_cat = train_fe[GRAPH_CAT_COLS].astype(str).copy()
graph_test_cat  = test_fe[GRAPH_CAT_COLS].astype(str).copy()
graph_all_cat = pd.concat([graph_train_cat, graph_test_cat], axis=0, ignore_index=True)

try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False, dtype=np.float32)

X_graph_cat = ohe.fit_transform(graph_all_cat).astype(np.float32)

graph_train_num = train_fe[GRAPH_NUM_COLS].copy()
graph_test_num  = test_fe[GRAPH_NUM_COLS].copy()

for c in GRAPH_NUM_COLS:
    graph_train_num[c] = pd.to_numeric(graph_train_num[c], errors="coerce").fillna(0).astype(np.float32)
    graph_test_num[c]  = pd.to_numeric(graph_test_num[c], errors="coerce").fillna(0).astype(np.float32)

graph_scaler = StandardScaler()
X_graph_num_train = graph_scaler.fit_transform(graph_train_num.values.astype(np.float32)).astype(np.float32)
X_graph_num_test  = graph_scaler.transform(graph_test_num.values.astype(np.float32)).astype(np.float32)

X_graph_num = np.vstack([X_graph_num_train, X_graph_num_test]).astype(np.float32)
X_graph_num *= GRAPH_NUMERIC_MULTIPLIER

X_graph = np.concatenate([X_graph_cat, X_graph_num], axis=1).astype(np.float32)

print("Building cuML KNN edges on:")
print(" - OHE of 16 base categorical features")
print(" - plus standardized 3 numerics multiplied by", GRAPH_NUMERIC_MULTIPLIER)
print("Graph categorical matrix shape:", X_graph_cat.shape)
print("Graph numeric matrix shape    :", X_graph_num.shape)
print("Graph final matrix shape      :", X_graph.shape)

knn = NearestNeighbors(n_neighbors=K)
knn.fit(X_graph)
_, idx = knn.kneighbors(X_graph)

neighbors = idx.astype(np.int32)

print("neighbors:", neighbors.shape)

del X_graph, X_graph_cat, X_graph_num, idx, knn
gc.collect()

# ============================================================
# BASE DATA ON CPU
# ============================================================
x_num_cpu = torch.tensor(Xn_all, dtype=torch.float32).pin_memory()
x_cat_cpu = torch.tensor(Xc_all, dtype=torch.long).pin_memory()
y_all = np.concatenate([y, np.full(n_test, -1, np.float32)]).astype(np.float32)
y_cpu = torch.tensor(y_all, dtype=torch.float32).pin_memory()

print("CPU tensors ready:")
print("x_num:", tuple(x_num_cpu.shape), "x_cat:", tuple(x_cat_cpu.shape), "y:", tuple(y_cpu.shape))

# ============================================================
# MODEL: Cat Embeddings + GraphSAGE (FAST 2-LAYER VERSION)
# ============================================================
def emb_dim_from_card(card: int) -> int:
    d = int(round(1.8 * (card ** 0.25)))
    return int(np.clip(d, 4, 24))

class CatEmbed(nn.Module):
    def __init__(self, cardinals):
        super().__init__()
        self.embs = nn.ModuleList()
        out_dim = 0
        for card in cardinals:
            card = max(2, int(card))
            d = emb_dim_from_card(card)
            self.embs.append(nn.Embedding(card, d))
            out_dim += d
        self.out_dim = out_dim

        for e in self.embs:
            nn.init.normal_(e.weight, mean=0.0, std=0.02)

    def forward(self, x_cat):
        zs = [emb(x_cat[:, j]) for j, emb in enumerate(self.embs)]
        return torch.cat(zs, dim=1)

class SAGEWithCats(nn.Module):
    def __init__(self, num_in, cat_cardinals, hidden=128, dropout=0.20):
        super().__init__()
        self.cat = CatEmbed(cat_cardinals)
        in_dim = num_in + self.cat.out_dim

        self.lin_in = nn.Linear(in_dim, hidden)
        self.conv1 = SAGEConv(hidden, hidden)
        self.conv2 = SAGEConv(hidden, hidden)

        self.norm1 = nn.LayerNorm(hidden)
        self.norm2 = nn.LayerNorm(hidden)

        self.dropout = dropout

        self.head = nn.Sequential(
            nn.Linear(hidden, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, data):
        zc = self.cat(data.x_cat)
        x = torch.cat([data.x_num, zc], dim=1)

        x = F.relu(self.lin_in(x))
        x = F.dropout(x, p=self.dropout, training=self.training)

        x1 = F.relu(self.norm1(self.conv1(x, data.edge_index)))
        x1 = F.dropout(x1, p=self.dropout, training=self.training)
        x = x + 0.5 * x1

        x2 = F.relu(self.norm2(self.conv2(x, data.edge_index)))
        x2 = F.dropout(x2, p=self.dropout, training=self.training)
        x = x + 0.5 * x2

        return self.head(x).squeeze(-1)

class SmoothBCE(nn.Module):
    def __init__(self, eps=0.01):
        super().__init__()
        self.eps = eps

    def forward(self, logits, targets):
        t = targets * (1 - self.eps) + 0.5 * self.eps
        return F.binary_cross_entropy_with_logits(logits, t)

# ============================================================
# FAST SUBGRAPH SAMPLER
# ============================================================
global_pos = np.full(n_all, -1, dtype=np.int32)

def build_subgraph_from_seeds_fast(
    seed_nodes,
    neighbors,
    x_num_cpu,
    x_cat_cpu,
    y_cpu,
    fanouts,
    device,
    offset=0,
):
    seed_nodes = np.asarray(seed_nodes, dtype=np.int32)

    frontier = seed_nodes
    collected = [seed_nodes]

    for hop, fanout in enumerate(fanouts):
        nbr = neighbors[frontier]  # [num_frontier, K]
        start = (offset + hop) % nbr.shape[1]
        cols = (np.arange(fanout) + start) % nbr.shape[1]
        nbr = nbr[:, cols]

        frontier = np.unique(nbr.reshape(-1))
        collected.append(frontier)

    nodes = np.unique(np.concatenate(collected))
    m = len(nodes)

    global_pos[nodes] = np.arange(m, dtype=np.int32)
    seed_local = global_pos[seed_nodes]

    sub_nbr = neighbors[nodes]
    dst_local_all = global_pos[sub_nbr]
    mask = dst_local_all >= 0

    src_local = np.repeat(np.arange(m, dtype=np.int64), sub_nbr.shape[1])[mask.reshape(-1)]
    dst_local = dst_local_all[mask].astype(np.int64)

    edge_index = torch.tensor(
        np.vstack([src_local, dst_local]),
        dtype=torch.long,
        device=device
    )

    batch = Data(
        x_num=x_num_cpu[nodes].to(device, non_blocking=True),
        x_cat=x_cat_cpu[nodes].to(device, non_blocking=True),
        y=y_cpu[nodes].to(device, non_blocking=True),
        edge_index=edge_index,
    )
    batch.seed_local = torch.tensor(seed_local, dtype=torch.long, device=device)

    global_pos[nodes] = -1
    return batch

def iterate_seed_batches(seed_nodes, batch_size, shuffle):
    arr = np.asarray(seed_nodes, dtype=np.int32).copy()
    if shuffle:
        np.random.shuffle(arr)
    for i in range(0, len(arr), batch_size):
        yield arr[i:i + batch_size]

@torch.no_grad()
def predict_seed_nodes_fast(
    model,
    seed_nodes,
    neighbors,
    x_num_cpu,
    x_cat_cpu,
    y_cpu,
    fanouts,
    batch_size,
    device,
    offset=0,
):
    model.eval()
    out = np.zeros(len(seed_nodes), dtype=np.float32)
    pos = 0

    for batch_seed_nodes in iterate_seed_batches(seed_nodes, batch_size, shuffle=False):
        batch = build_subgraph_from_seeds_fast(
            seed_nodes=batch_seed_nodes,
            neighbors=neighbors,
            x_num_cpu=x_num_cpu,
            x_cat_cpu=x_cat_cpu,
            y_cpu=y_cpu,
            fanouts=fanouts,
            device=device,
            offset=offset,
        )

        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(USE_AMP and DEVICE == "cuda")):
            logits = model(batch)

        probs = torch.sigmoid(logits[batch.seed_local]).float().cpu().numpy()
        out[pos:pos + len(batch_seed_nodes)] = probs.astype(np.float32)
        pos += len(batch_seed_nodes)

        del batch, logits, probs

    return out

# ============================================================
# CV TRAIN
# ============================================================
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof = np.zeros(n_train, dtype=np.float32)
pred_test = np.zeros(n_test, dtype=np.float32)

loss_fn = SmoothBCE(eps=0.01)
test_nodes = np.arange(n_train, n_all, dtype=np.int32)

for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(n_train), y), 1):
    print(f"\n================ Fold {fold}/{N_FOLDS} ================")

    model = SAGEWithCats(
        num_in=Xn_all.shape[1],
        cat_cardinals=cat_cardinals,
        hidden=128,
        dropout=0.20,
    ).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=3e-4)
    scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and DEVICE == "cuda"))

    best_auc = -1.0
    best_state = None
    bad = 0
    t_fold = time.time()

    for epoch in range(1, EPOCHS + 1):
        model.train()
        losses = []
        offset = epoch % K

        for batch_seed_nodes in iterate_seed_batches(tr_idx, BATCH_SIZE, shuffle=True):
            batch = build_subgraph_from_seeds_fast(
                seed_nodes=batch_seed_nodes,
                neighbors=neighbors,
                x_num_cpu=x_num_cpu,
                x_cat_cpu=x_cat_cpu,
                y_cpu=y_cpu,
                fanouts=FANOUTS,
                device=DEVICE,
                offset=offset,
            )

            opt.zero_grad(set_to_none=True)

            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(USE_AMP and DEVICE == "cuda")):
                logits = model(batch)
                loss = loss_fn(logits[batch.seed_local], batch.y[batch.seed_local])

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()

            losses.append(loss.item())
            del batch, logits, loss

        do_val = (epoch % VAL_EVERY == 0) or (epoch == 1)

        if do_val:
            val_pred = predict_seed_nodes_fast(
                model=model,
                seed_nodes=va_idx,
                neighbors=neighbors,
                x_num_cpu=x_num_cpu,
                x_cat_cpu=x_cat_cpu,
                y_cpu=y_cpu,
                fanouts=FANOUTS,
                batch_size=INFER_BATCH_SIZE,
                device=DEVICE,
                offset=offset,
            )
            auc = roc_auc_score(y[va_idx], val_pred)

            print(f"epoch {epoch:03d} | loss {np.mean(losses):.5f} | val_auc {auc:.6f}")

            if auc > best_auc + 1e-6:
                best_auc = auc
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                bad = 0
            else:
                bad += 1
                if epoch >= MIN_EPOCHS and bad >= PATIENCE:
                    break

    model.load_state_dict(best_state)
    model.eval()

    val_pred = predict_seed_nodes_fast(
        model=model,
        seed_nodes=va_idx,
        neighbors=neighbors,
        x_num_cpu=x_num_cpu,
        x_cat_cpu=x_cat_cpu,
        y_cpu=y_cpu,
        fanouts=FANOUTS,
        batch_size=INFER_BATCH_SIZE,
        device=DEVICE,
        offset=0,
    )
    oof[va_idx] = val_pred.astype(np.float32)

    test_pred = predict_seed_nodes_fast(
        model=model,
        seed_nodes=test_nodes,
        neighbors=neighbors,
        x_num_cpu=x_num_cpu,
        x_cat_cpu=x_cat_cpu,
        y_cpu=y_cpu,
        fanouts=FANOUTS,
        batch_size=INFER_BATCH_SIZE,
        device=DEVICE,
        offset=0,
    )
    pred_test += test_pred.astype(np.float32) / N_FOLDS

    print(f"[Fold {fold}] best AUC: {best_auc:.6f} | fold time {time.time() - t_fold:.1f}s")

    del model, opt, scaler, best_state, val_pred, test_pred
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

cv_auc = roc_auc_score(y, oof)
print("\n====================")
print("OOF CV AUC:", cv_auc)
print("====================")

np.save(f"oof_gnn_v{VER}.npy", oof.astype(np.float32))
np.save(f"pred_gnn_v{VER}.npy", pred_test.astype(np.float32))
print(f"Saved: oof_gnn_v{VER}.npy")
print(f"Saved: pred_gnn_v{VER}.npy")

train: (594194, 21) test: (254655, 20)
Using same simple 25 node features:
NUM_COLS: ['tenure', 'MonthlyCharges', 'TotalCharges']
CAT_COLS: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'tenure__cat', 'MonthlyCharges__cat', 'TotalCharges__cat', 'tenure__is_rare', 'MonthlyCharges__is_rare', 'TotalCharges__is_rare']
Xn_all: (848849, 3) Xc_all: (848849, 22)
Building cuML KNN edges on:
 - OHE of 16 base categorical features
 - plus standardized 3 numerics multiplied by 3.0
Graph categorical matrix shape: (848849, 43)
Graph numeric matrix shape    : (848849, 3)
Graph final matrix shape      : (848849, 46)
neighbors: (848849, 8)
CPU tensors ready:
x_num: (848849, 3) x_cat: (848849, 22) y: (848849,)

================ Fold 1/5 ================
epoch 001 | loss 0.38312 | val_auc 0.912566

# Write Submission CSV

In [3]:
sub  = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv")
sub.Churn = pred_test
sub.to_csv(f"submission_gnn_v{VER}.csv",index=False)
sub.head()

,id,Churn
0,594194,0.157202
1,594195,0.005663
2,594196,0.124597
3,594197,0.009370
4,594198,0.589746


# GPU Hill Climb
In order to determine if this new model is diverse and helpful to our ensemble, we can run GPU hill climbing with this model added to our collection of previous models. Previously, I made 3 models [here][1]: PyTorch MLP, XGBoost, and Logistic Regression w/ Target Encoding. We now load those 3 models and add our GNN model and run hill climbing. 

We see that hill climbing selects GNN below indicating that our new model is diverse and helpful to our ensemble. Hooray!

[1]: https://www.kaggle.com/code/cdeotte/chatgpt-vibe-coding-3xgpu-models-cv-0-9178

In [4]:
# ============================================================
# GPU HILL CLIMBING ENSEMBLE (AUC, USE RANKS)
# - Converts every OOF and TEST prediction to global normalized ranks
# - Blends directly in rank space 
# - Starts from best single model (or FIRST if provided)
# - Repeatedly tries adding every model with weight grid search
# - Uses GPU approximate AUC during search, exact CPU AUC for final check
# ============================================================

import os
import gc
import numpy as np
import pandas as pd
import cupy as cp

from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata

# -----------------------------
# CONFIG
# -----------------------------
TRAIN_CSV  = "/kaggle/input/competitions/playground-series-s6e3/train.csv"
TEST_CSV   = "/kaggle/input/competitions/playground-series-s6e3/test.csv"
TARGET_COL = "Churn"
ID_COL     = "id"
VER        = 1

# Optional: force starting model by name, else use best single
FIRST = None

USE_NEGATIVE_WGT = True
MAX_MODELS = 1000
TOL = 1e-5

start = -0.50 if USE_NEGATIVE_WGT else 0.01
stop  =  0.50
step  =  0.01

# NOTE: define VER before running
SUBMISSION_PATH = f"submission_hillclimb_v{VER}.csv"
WEIGHTS_PATH    = f"ensemble_weights_v{VER}.csv"

# -----------------------------
# YOUR MODELS
# -----------------------------
FILES = [
    ("gnn_v1909",     "oof_gnn_v1909.npy"),
    ("nn_v312",     "/kaggle/input/notebooks/cdeotte/chatgpt-vibe-coding-3xgpu-models-cv-0-9178/oof_nn_v312.npy"),
    ("xgb_v102",     "/kaggle/input/notebooks/cdeotte/chatgpt-vibe-coding-3xgpu-models-cv-0-9178/oof_pred_proba_v102.npy"),
    ("logreg_v200",     "/kaggle/input/notebooks/cdeotte/chatgpt-vibe-coding-3xgpu-models-cv-0-9178/oof_tepair_logit3_v200.npy"),
]

# -----------------------------
# HELPERS
# -----------------------------
def to_binary_y(df, col):
    y = df[col]
    if y.dtype == "object" or str(y.dtype).startswith("string"):
        y = y.map({"No": 0, "Yes": 1})
    y = pd.to_numeric(y, errors="coerce")
    if y.isna().any():
        raise ValueError(f"Could not parse target column '{col}' to binary.")
    y = y.astype(np.int32).to_numpy()
    return y

def auc_cpu(y_true, y_pred):
    return float(roc_auc_score(y_true, y_pred))

def rank01(x):
    x = np.asarray(x, dtype=np.float64).reshape(-1)
    r = rankdata(x, method="average")
    if len(r) == 1:
        return np.array([0.5], dtype=np.float64)
    return (r - 1.0) / (len(r) - 1.0)

def derive_test_path(oof_path):
    cand1 = oof_path.replace("oof", "pred", 1)
    cand2 = oof_path.replace("oof", "test", 1)
    if os.path.exists(cand1):
        return cand1
    if os.path.exists(cand2):
        return cand2
    raise FileNotFoundError(f"Could not find pred/test file for: {oof_path}")

def load_preds_1d(path, expected_len, csv_col=TARGET_COL):
    if path.endswith(".csv"):
        df = pd.read_csv(path)
        if csv_col not in df.columns:
            raise ValueError(f"{path} missing column '{csv_col}'")
        arr = df[csv_col].to_numpy()
    else:
        arr = np.load(path)

    arr = np.asarray(arr, dtype=np.float64)

    if arr.ndim == 2:
        if arr.shape[1] == 1:
            arr = arr[:, 0]
        elif arr.shape[0] == expected_len:
            arr = arr.mean(axis=1)
        else:
            raise ValueError(f"Unexpected shape for {path}: {arr.shape}")

    arr = arr.reshape(-1)

    if len(arr) != expected_len:
        raise ValueError(f"Length mismatch for {path}: got {len(arr)}, expected {expected_len}")

    return arr

def multiple_roc_auc_scores(y_gpu, preds_gpu):
    """
    Fast approximate AUC on GPU for many candidate columns at once.
    Ties are approximate, which is okay for hill-climbing search.
    """
    n_pos = cp.sum(y_gpu)
    n_neg = y_gpu.size - n_pos
    ranks = cp.argsort(cp.argsort(preds_gpu, axis=0), axis=0) + 1
    aucs = (cp.sum(ranks[y_gpu == 1, :], axis=0) - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)
    return aucs

# ============================================================
# 1) LOAD DATA
# ============================================================
train = pd.read_csv(TRAIN_CSV)
test  = pd.read_csv(TEST_CSV)

y = to_binary_y(train, TARGET_COL)
N = len(train)
M = len(test)

print(f"Train rows: {N:,} | Pos rate: {y.mean():.6f}")
print(f"Test rows : {M:,}")

# ============================================================
# 2) LOAD OOF / TEST PREDICTIONS
# ============================================================
names = []
x_train = []
x_test = []

print("\nLoading predictions...")
for name, oof_path in FILES:
    test_path = derive_test_path(oof_path)

    oof = load_preds_1d(oof_path, N)
    pred = load_preds_1d(test_path, M)

    oof = rank01(oof)
    pred = rank01(pred)

    names.append(name)
    x_train.append(oof)
    x_test.append(pred)

    print(f"{name:18s} | {oof_path} | {test_path}")

x_train = np.column_stack(x_train)
x_test  = np.column_stack(x_test)

print("\nOOF matrix :", x_train.shape)
print("TEST matrix:", x_test.shape)

# ============================================================
# 3) SINGLE MODEL AUC
# ============================================================
print("\nSingle-model AUC:")
single_aucs = []
for k, name in enumerate(names):
    s = auc_cpu(y, x_train[:, k])
    single_aucs.append(s)
    print(f"{name:18s} AUC = {s:.6f}")

best_idx = int(np.argmax(single_aucs))
best_auc = float(single_aucs[best_idx])
print(f"\nBest single: {names[best_idx]} (AUC={best_auc:.6f})")

# ============================================================
# 4) START MODEL
# ============================================================
if FIRST is None:
    start_idx = best_idx
else:
    if FIRST not in names:
        raise ValueError(f"FIRST='{FIRST}' not found in model list.")
    start_idx = names.index(FIRST)

start_auc = auc_cpu(y, x_train[:, start_idx])
print(f"Starting from: {names[start_idx]} (AUC={start_auc:.6f})")

# ============================================================
# 5) PREP GPU
# ============================================================
x_gpu = cp.asarray(x_train, dtype=cp.float32)
y_gpu = cp.asarray(y, dtype=cp.int8)

ww = cp.arange(start, stop + 1e-9, step, dtype=cp.float32)

best_ensemble = x_gpu[:, start_idx].copy()
best_score = float(start_auc)

models = [start_idx]
weights_step = []

# ============================================================
# 6) HILL CLIMB
# ============================================================
print("\nStarting hill climb...\n")

for it in range(1_000_000):
    if len(set(models)) >= MAX_MODELS:
        print(f"Reached MAX_MODELS={MAX_MODELS}")
        break

    base = best_ensemble[:, None] * (1.0 - ww)[None, :]

    best_it_score = -1.0
    best_it_idx = -1
    best_it_w = None
    best_it_ens = None

    for k in range(len(names)):
        cand = base + x_gpu[:, k][:, None] * ww[None, :]
        aucs = multiple_roc_auc_scores(y_gpu, cand)

        j = int(cp.argmax(aucs).item())
        s = float(aucs[j].item())

        if s > best_it_score:
            best_it_score = s
            best_it_idx = k
            best_it_w = float(ww[j].item())
            best_it_ens = cand[:, j]

        del cand, aucs

    improve = best_it_score - best_score
    if improve < TOL:
        print(f"Stopped: improvement {improve:.8f} < TOL={TOL}")
        break

    print(f"{it:3d}  AUC {best_it_score:.6f} | add {names[best_it_idx]} | w = {best_it_w:.3f}")

    models.append(best_it_idx)
    weights_step.append(best_it_w)
    best_ensemble = best_it_ens
    best_score = best_it_score

    del base
    gc.collect()
    cp._default_memory_pool.free_all_blocks()

# ============================================================
# 7) FINAL EXACT OOF AUC
# ============================================================
final_oof = best_ensemble.get()
final_auc = auc_cpu(y, final_oof)
print(f"\nFinal OOF AUC (exact CPU): {final_auc:.6f}")

# ============================================================
# 8) STEP WEIGHTS -> EFFECTIVE WEIGHTS
# ============================================================
wgt = np.array([1.0], dtype=np.float64)
for w in weights_step:
    wgt = wgt * (1.0 - w)
    wgt = np.append(wgt, w)

dfw = (
    pd.DataFrame({
        "model": [names[m] for m in models],
        "weight": wgt
    })
    .groupby("model", as_index=False)["weight"].sum()
    .sort_values("weight", ascending=False)
    .reset_index(drop=True)
)

print("\nFinal weights:")
print(dfw.to_string(index=False))
print("\nWeight sum:", dfw["weight"].sum())

dfw.to_csv(WEIGHTS_PATH, index=False)
print(f"Saved weights: {WEIGHTS_PATH}")

# ============================================================
# 9) SANITY CHECK OOF
# ============================================================
blend_oof = np.zeros(N, dtype=np.float64)
for _, row in dfw.iterrows():
    k = names.index(row["model"])
    blend_oof += x_train[:, k] * float(row["weight"])

print(f"Sanity OOF AUC (exact CPU): {auc_cpu(y, blend_oof):.6f}")

# ============================================================
# 10) TEST BLEND + SUBMISSION
# ============================================================
blend_test = np.zeros(M, dtype=np.float64)
for _, row in dfw.iterrows():
    k = names.index(row["model"])
    blend_test += x_test[:, k] * float(row["weight"])

if ID_COL in test.columns:
    sub = pd.DataFrame({ID_COL: test[ID_COL].values, TARGET_COL: blend_test})
else:
    sub = pd.DataFrame({TARGET_COL: blend_test})

sub.to_csv(SUBMISSION_PATH, index=False)
print(f"\nSaved submission: {SUBMISSION_PATH}")
print(sub.head())

print("\nDone.")

Train rows: 594,194 | Pos rate: 0.225208
Test rows : 254,655

Loading predictions...
gnn_v1909          | oof_gnn_v1909.npy | pred_gnn_v1909.npy
nn_v312            | /kaggle/input/notebooks/cdeotte/chatgpt-vibe-coding-3xgpu-models-cv-0-9178/oof_nn_v312.npy | /kaggle/input/notebooks/cdeotte/chatgpt-vibe-coding-3xgpu-models-cv-0-9178/pred_nn_v312.npy
xgb_v102           | /kaggle/input/notebooks/cdeotte/chatgpt-vibe-coding-3xgpu-models-cv-0-9178/oof_pred_proba_v102.npy | /kaggle/input/notebooks/cdeotte/chatgpt-vibe-coding-3xgpu-models-cv-0-9178/test_pred_proba_v102.npy
logreg_v200        | /kaggle/input/notebooks/cdeotte/chatgpt-vibe-coding-3xgpu-models-cv-0-9178/oof_tepair_logit3_v200.npy | /kaggle/input/notebooks/cdeotte/chatgpt-vibe-coding-3xgpu-models-cv-0-9178/pred_tepair_logit3_v200.npy

OOF matrix : (594194, 4)
TEST matrix: (254655, 4)

Single-model AUC:
gnn_v1909          AUC = 0.915442
nn_v312            AUC = 0.916369
xgb_v102           AUC = 0.916601
logreg_v200        AUC = 0.